
## Build page-linked legal chunks
This cell converts the page inventory into smaller text chunks for retrieval. It detects GDPR articles, EDPB chapters and subsections, and numbered paragraphs, then stores each chunk with its document name, PDF page, section, and paragraph. These metadata fields help the application retrieve relevant passages and show where its evidence came from.

The chunks are split at detected headings and paragraph boundaries while retaining the section and paragraph labels. The resulting `legal_chunks.csv` becomes the source for later search and evaluation.

In [1]:
from pathlib import Path
import csv
import re

project_dir = Path.home() / "Desktop/LexiSyncAI"
inventory_path = project_dir / "data/processed/page_inventory.csv"
output_path = project_dir / "data/processed/legal_chunks.csv"

with inventory_path.open(encoding="utf-8", newline="") as file:
    pages = list(csv.DictReader(file))

pages.sort(key=lambda row: (row["document"], int(row["page_number"])))

article_pattern = re.compile(r"^\s*Article\s+(\d+[A-Za-z]?)\b", re.IGNORECASE)
chapter_pattern = re.compile(r"^\s*(CHAPTER\s+\d+\b.*)", re.IGNORECASE)
subsection_pattern = re.compile(r"^\s*(\d+\.\d+\s*[-–].*)")
paragraph_pattern = re.compile(r"^\s*(\d{1,3})\.\s+")

chunks = []
section_by_document = {}
paragraph_by_document = {}

for page in pages:
    document = page["document"]
    page_number = int(page["page_number"])
    section = section_by_document.get(document, "Unclassified section")
    paragraph = paragraph_by_document.get(document, "")
    lines = []

    def save_chunk():
        text = "\n".join(lines).strip()
        if text:
            chunks.append({
                "chunk_id": f"{document}::p{page_number}::c{len(chunks) + 1}",
                "document": document,
                "page_number": page_number,
                "section": section,
                "paragraph": paragraph,
                "text": text,
            })

    for line in page["text"].splitlines():
        line = line.strip()
        if not line:
            continue

        article_match = article_pattern.match(line)
        chapter_match = chapter_pattern.match(line)
        subsection_match = subsection_pattern.match(line)

        if article_match:
            save_chunk()
            section = f"Article {article_match.group(1)}"
            paragraph = ""
            lines = [line]
            continue

        if chapter_match:
            save_chunk()
            section = chapter_match.group(1)
            paragraph = ""
            lines = [line]
            continue

        if subsection_match:
            save_chunk()
            section = subsection_match.group(1)
            paragraph = ""
            lines = [line]
            continue

        paragraph_match = paragraph_pattern.match(line)
        if paragraph_match:
            # Keep the section heading with its first paragraph.
            if lines and not paragraph:
                pass
            else:
                save_chunk()
                lines = []

            paragraph = f"Paragraph {paragraph_match.group(1)}"

        lines.append(line)

    save_chunk()
    section_by_document[document] = section
    paragraph_by_document[document] = paragraph

output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "chunk_id",
            "document",
            "page_number",
            "section",
            "paragraph",
            "text",
        ],
    )
    writer.writeheader()
    writer.writerows(chunks)

print(f"Created {len(chunks)} chunks: {output_path}")
print("\nSample chunks:")
for chunk in chunks:
    if chunk["section"] == "Article 83":
        print(
            f"\n{chunk['document']} | PDF page {chunk['page_number']} "
            f"| {chunk['section']} | {chunk['paragraph']}"
        )
        print(chunk["text"][:700])

Created 723 chunks: /Users/shashankbangaru/Desktop/LexiSyncAI/data/processed/legal_chunks.csv

Sample chunks:

edpb_fines_guidelines.pdf | PDF page 9 | Article 83 | 
Article 83(1) GDPR.
Step 3   Evaluating aggravating and mitigating circumstances related to past or present behaviour of
the controller/processor and increasing or decreasing the fine accordingly. (Chapter 5)
Step 4    Identifying the relevant legal maximums for the different processing operations. Increases
applied in previous or next steps cannot exceed this amount. (Chapter 6)
Step 5   Analysing whether the final amount of the calculated fine meets the requirements of
effectiveness, dissuasiveness and proportionality, as required by Article 83(1) GDPR, and
increasing or decreasing the fine accordingly. (Chapter 7)

edpb_fines_guidelines.pdf | PDF page 9 | Article 83 | Paragraph 21
ARTICLE 83(3) GDPR
21.   Before being able to calculate a fine based on the methodology of these guidelines, it is important to first
conside

## Check Article 83 chunks and page citations
This cell checks how GDPR Article 83 was divided into chunks. It prints each matching chunk’s PDF page and paragraph label so I can verify that the article’s content remains associated with useful citation metadata after chunking.

In [2]:
from pathlib import Path
import csv

chunks_path = (
    Path.home()
    / "Desktop/LexiSyncAI/data/processed/legal_chunks.csv"
)

with chunks_path.open(encoding="utf-8", newline="") as file:
    chunks = list(csv.DictReader(file))

for chunk in chunks:
    if (
        chunk["document"] == "gdpr_regulation.pdf"
        and chunk["section"] == "Article 83"
    ):
        print(
            f"\nPDF page {chunk['page_number']} | "
            f"{chunk['paragraph']}"
        )
        print(chunk["text"][:900])


PDF page 82 | Paragraph 1
Article 83
General conditions for imposing administrative fines
1.   Each supervisory authority shall ensure that the imposition of administrative fines pursuant to this Article in
respect of infringements of this Regulation referred to in paragraphs 4, 5 and 6 shall in each individual case be effective,
proportionate and dissuasive.

PDF page 82 | Paragraph 2
2.    Administrative fines shall, depending on the circumstances of each individual case, be imposed in addition to, or
instead of, measures referred to in points (a) to (h) and  (j) of Article 58(2). When deciding whether to impose an
administrative fine and deciding on the amount of the administrative fine in each individual case due regard shall be
given to the following:
(a) the nature, gravity and duration of the infringement taking into account the nature scope or purpose of the
processing concerned as well as the number of data subjects affected and the level of damage suffered by them;
(b) the i

## Check chunk coverage across all documents
This cell summarizes the number of chunks and detected sections for each document. It provides a quick check that both PDFs were processed and shows which sections produced the most chunks. Unexpected section labels can reveal where the heading-detection rules need review.

In [3]:
from pathlib import Path
import csv
from collections import Counter

chunks_path = (
    Path.home()
    / "Desktop/LexiSyncAI/data/processed/legal_chunks.csv"
)

with chunks_path.open(encoding="utf-8", newline="") as file:
    chunks = list(csv.DictReader(file))

for document in sorted({chunk["document"] for chunk in chunks}):
    document_chunks = [c for c in chunks if c["document"] == document]
    section_counts = Counter(c["section"] for c in document_chunks)

    print(f"\n=== {document} ===")
    print(f"Chunks: {len(document_chunks)}")
    print(f"Sections detected: {len(section_counts)}")

    for section, count in section_counts.most_common(20):
        print(f"  {section}: {count} chunks")


=== edpb_fines_guidelines.pdf ===
Chunks: 233
Sections detected: 78
  Article 83: 24 chunks
  3.1 - One sanctionable conduct: 23 chunks
  6.2 - Determining the undertaking’s turnover and corporate liability: 17 chunks
  5.4 - Previous infringements by the controller or processor: 14 chunks
  4.3 - Turnover of the undertaking with a view to imposing an effective, dissuasive and: 9 chunks
  5.9 - Other aggravating and mitigating circumstances: 6 chunks
  1.2 - Objective: 5 chunks
  1.3 - Scope: 5 chunks
  5.3 - Degree of responsibility of the controller or processor: 5 chunks
  CHAPTER 8 – FLEXIBILITY AND REGULAR EVALUATION: 5 chunks
  Unclassified section: 4 chunks
  4.2 - Seriousness of the infringement in each individual case: 4 chunks
  5.2 - Actions taken by controller or processor to mitigate damage suffered by data subjects: 4 chunks
  5.7 - Compliance with measures previously ordered with regard to the same subject matter: 4 chunks
  5.8 - Adherence to approved codes of conduct 

One thing to review in the output: EDPB chunks are also labeled “Article 83.” That likely happens because the chunking pattern treats any line beginning with “Article 83” as a GDPR article, even when it appears as a reference inside the EDPB guidelines. Check those chunks before relying on their section labels for citations.
